[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_11/notebook_11_1_differential_privacy.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 11.1: Differential Privacy for Healthcare AI

**Chapter 11: Privacy, Security, and Trustworthy AI**

## Introduction: The Privacy Paradox

Healthcare AI requires large datasets, but sharing patient data risks privacy:

**Traditional approach**: Remove names, dates → "de-identified" data

**Problem**: Re-identification attacks succeed 63-87% of the time
- Netflix Prize: "Anonymous" movie ratings re-identified via IMDb
- AOL search logs: User 4417749 identified as Thelma Arnold via Google searches  
- Hospital discharge data: 87% re-identified using ZIP+birthdate+gender

**Solution**: **Differential Privacy (DP)** - mathematical guarantee that no individual can be identified from model outputs.

### What is Differential Privacy?

**Definition**: A mechanism $\mathcal{M}$ satisfies $\varepsilon$-differential privacy if for all datasets $D$ and $D'$ differing in one record:

$$\Pr[\mathcal{M}(D) \in S] \leq e^{\varepsilon} \cdot \Pr[\mathcal{M}(D') \in S]$$

**Intuition**: Whether Alice's data is included or not barely affects the output.

**Privacy budget ($\varepsilon$)**:
- $\varepsilon \leq 1$: Strong privacy
- $1 < \varepsilon \leq 8$: Moderate privacy
- $\varepsilon > 10$: Weak privacy (little meaningful protection)

### Why DP Matters for Healthcare

1. **Provable guarantee**: Mathematics, not heuristics
2. **Composability**: Know cumulative privacy loss over multiple queries
3. **Regulatory context**: a quantifiable guarantee can support (not replace) GDPR/HIPAA compliance work
4. **Trust**: Patients more willing to share data with DP guarantees

---

## Part 1: DP Basics - Adding Noise

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print("\nThis notebook implements differential privacy for healthcare AI.\n")

✓ Libraries imported

This notebook implements differential privacy for healthcare AI.



### Example 1: Counting with Laplace Noise

In [2]:
# Create synthetic patient data
n_patients = 10000
diabetes = np.random.binomial(1, 0.12, n_patients)  # 12% have diabetes

true_count = diabetes.sum()

print(f"True count of diabetic patients: {true_count}")
print(f"True prevalence: {true_count/n_patients:.1%}\n")

# Non-private query
print("NON-PRIVATE QUERY:")
print(f"  Answer: {true_count} patients")
print(f"  Privacy: ❌ Exact count reveals information\n")

# DP query with Laplace mechanism
def laplace_mechanism(true_value, sensitivity, epsilon):
    """
    Add Laplace noise for epsilon-DP.

    Parameters:
    - true_value: Actual query result
    - sensitivity: Maximum change from adding/removing one record
    - epsilon: Privacy budget
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale)
    return true_value + noise

# For counting, sensitivity = 1 (one person changes count by ±1)
epsilon_values = [10.0, 1.0, 0.1]

print("DIFFERENTIAL PRIVACY QUERIES:\n")
for eps in epsilon_values:
    noisy_count = laplace_mechanism(true_count, sensitivity=1, epsilon=eps)

    privacy_level = "Strong" if eps <= 1 else "Moderate" if eps <= 8 else "Weak"

    print(f"ε = {eps}:")
    print(f"  Noisy count: {noisy_count:.1f} patients")
    print(f"  Error: {abs(noisy_count - true_count):.1f}")
    print(f"  Privacy: {privacy_level}")
    print()

print("💡 Trade-off: Stronger privacy (smaller ε) → More noise → Less accurate")

True count of diabetic patients: 1145
True prevalence: 11.5%

NON-PRIVATE QUERY:
  Answer: 1145 patients
  Privacy: ❌ Exact count reveals information

DIFFERENTIAL PRIVACY QUERIES:

ε = 10.0:
  Noisy count: 1145.0 patients
  Error: 0.0
  Privacy: Weak

ε = 1.0:
  Noisy count: 1144.6 patients
  Error: 0.4
  Privacy: Strong

ε = 0.1:
  Noisy count: 1134.6 patients
  Error: 10.4
  Privacy: Strong

💡 Trade-off: Stronger privacy (smaller ε) → More noise → Less accurate


### Visualize Privacy-Utility Trade-off

In [3]:
# Run multiple queries at different privacy levels
epsilon_range = np.logspace(-1, 1.5, 20)  # 0.1 to ~30
errors = []

for eps in epsilon_range:
    # Average over 100 trials
    trial_errors = []
    for _ in range(100):
        noisy = laplace_mechanism(true_count, sensitivity=1, epsilon=eps)
        trial_errors.append(abs(noisy - true_count))
    errors.append(np.mean(trial_errors))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(epsilon_range, errors, 'o-', linewidth=2, markersize=6, color='darkblue')
ax.axvline(x=1.0, color='green', linestyle='--', linewidth=2, label='Strong Privacy (ε=1)')
ax.axvline(x=10.0, color='orange', linestyle='--', linewidth=2, label='Weak Privacy (ε=10)')

ax.set_xlabel('Privacy Budget (ε)', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Error (count)', fontsize=12, fontweight='bold')
ax.set_title('Differential Privacy: Privacy vs Accuracy Trade-off', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dp_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Observation: As ε decreases (stronger privacy), error increases.")
print("   ε=1 is a good balance for most healthcare applications.\n")


📊 Observation: As ε decreases (stronger privacy), error increases.
   ε=1 is a good balance for most healthcare applications.



## Part 2: DP Machine Learning - DP-SGD

Training ML models with differential privacy requires:
1. **Gradient clipping**: Limit each example's influence
2. **Noise addition**: Add Gaussian noise to gradients
3. **Privacy accounting**: Track cumulative privacy loss across epochs

In [4]:
# Create diabetes prediction dataset
def create_diabetes_dataset(n_samples=5000):
    data = []
    for i in range(n_samples):
        age = np.random.normal(55, 15)
        bmi = np.random.normal(28, 6)
        glucose = np.random.normal(110, 30)

        # Calculate diabetes risk
        risk = 0.05
        if age > 60:
            risk *= 2.0
        if bmi > 30:
            risk *= 2.5
        if glucose > 125:
            risk *= 3.0

        has_diabetes = np.random.random() < min(risk, 0.6)

        data.append({
            'age': np.clip(age, 20, 90),
            'bmi': np.clip(bmi, 15, 50),
            'glucose': np.clip(glucose, 60, 250),
            'diabetes': int(has_diabetes)
        })

    return pd.DataFrame(data)

df = create_diabetes_dataset(n_samples=5000)

print("Diabetes Dataset:")
print(f"  Samples: {len(df)}")
print(f"  Prevalence: {df['diabetes'].mean():.1%}\n")
df.head()

Diabetes Dataset:
  Samples: 5000
  Prevalence: 16.8%



,age,bmi,glucose,diabetes
0,57.416739,37.222827,106.288485,0
1,66.146983,30.287468,108.781813,0
2,66.292725,26.000110,97.840968,0
3,62.304410,36.601802,60.000000,0
4,54.196219,30.519932,60.000000,0


In [5]:
# Prepare data
X = df[['age', 'bmi', 'glucose']].values
y = df['diabetes'].values

# Split FIRST, then normalise with fixed, public reference values (typical adult age, BMI,
# fasting glucose). Computing mean/std from the data would let one patient shift everyone's
# features (outside any DP guarantee) and would leak test data into preprocessing.
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
REF_MEAN = np.array([55.0, 28.0, 110.0])   # age (y), BMI (kg/m²), glucose (mg/dL)
REF_SD   = np.array([15.0, 6.0, 30.0])
X_train = (X_train_raw - REF_MEAN) / REF_SD
X_test = (X_test_raw - REF_MEAN) / REF_SD

print(f"Training: {len(X_train)} | Test: {len(X_test)}")

Training: 4000 | Test: 1000


### Baseline: Non-Private Model

In [6]:
# Train standard logistic regression
model_baseline = LogisticRegression(random_state=42, max_iter=1000)
model_baseline.fit(X_train, y_train)

y_pred_baseline = model_baseline.predict(X_test)
y_prob_baseline = model_baseline.predict_proba(X_test)[:, 1]

print("BASELINE (No Privacy):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_baseline):.1%}")
print(f"  AUC: {roc_auc_score(y_test, y_prob_baseline):.3f}")
print(f"  Privacy: ❌ None (model may memorize training data)\n")

BASELINE (No Privacy):
  Accuracy: 83.1%
  AUC: 0.675
  Privacy: ❌ None (model may memorize training data)



### DP-SGD: Simplified Implementation

We'll implement a simplified version of DP-SGD for logistic regression.

> ⚠️ **Teaching simplification — read before trusting any ε below.** The `epsilon` passed to this function only sets the noise level of **one** gradient step. Real DP-SGD must (a) calibrate the noise to the sensitivity of the clipped-gradient *sum* and (b) **compose** the privacy loss over every step (here ≈1,900 steps). This code does neither: no privacy accountant is run, so "ε=1" below is a label for a noise setting, **not** an accounted end-to-end guarantee for the trained model. For real guarantees use a library with an accountant, e.g. [Opacus](https://opacus.ai) (RDP/PRV accountant) or TensorFlow Privacy.

In [7]:
def dp_logistic_regression(X_train, y_train, epsilon, delta, epochs=50, batch_size=64, learning_rate=0.1, clip_norm=1.0):
    """
    Train logistic regression with differential privacy.

    Parameters:
    - epsilon: Privacy budget
    - delta: Failure probability (choose δ ≪ 1/n)
    - clip_norm: Gradient clipping threshold
    """
    n_samples, n_features = X_train.shape

    # Initialize weights
    w = np.zeros(n_features)
    b = 0

    # Calculate noise scale
    # Simplified: σ² ≥ 2 * log(1.25/δ) * (clip_norm)² / ε²
    noise_scale = clip_norm * np.sqrt(2 * np.log(1.25 / delta)) / epsilon

    for epoch in range(epochs):
        # Shuffle data
        indices = np.random.permutation(n_samples)

        for i in range(0, n_samples, batch_size):
            batch_indices = indices[i:i+batch_size]
            X_batch = X_train[batch_indices]
            y_batch = y_train[batch_indices]

            # Forward pass
            z = X_batch @ w + b
            pred = 1 / (1 + np.exp(-z))

            # Compute gradients (per example)
            grad_w_batch = []
            grad_b_batch = []

            for j in range(len(X_batch)):
                error = pred[j] - y_batch[j]
                grad_w = error * X_batch[j]
                grad_b = error

                # Clip gradient (per example)
                grad_norm = np.sqrt(np.sum(grad_w**2) + grad_b**2)
                if grad_norm > clip_norm:
                    grad_w = grad_w * (clip_norm / grad_norm)
                    grad_b = grad_b * (clip_norm / grad_norm)

                grad_w_batch.append(grad_w)
                grad_b_batch.append(grad_b)

            # Average clipped gradients
            avg_grad_w = np.mean(grad_w_batch, axis=0)
            avg_grad_b = np.mean(grad_b_batch)

            # Add Gaussian noise
            noisy_grad_w = avg_grad_w + np.random.normal(0, noise_scale, size=n_features)
            noisy_grad_b = avg_grad_b + np.random.normal(0, noise_scale)

            # Update weights
            w -= learning_rate * noisy_grad_w
            b -= learning_rate * noisy_grad_b

    return w, b

print("✓ DP-SGD function defined\n")

✓ DP-SGD function defined



In [8]:
# Train DP models with different privacy budgets
epsilon_values = [10.0, 3.0, 1.0, 0.5]  # the sweep used in the chapter
delta = 1e-5  # rule of thumb: δ ≪ 1/n (here 1/n = 1/4000)

results = []

print("Training DP models...\n")

for eps in epsilon_values:
    w, b = dp_logistic_regression(
        X_train, y_train,
        epsilon=eps,
        delta=delta,
        epochs=30,
        batch_size=64,
        learning_rate=0.1,
        clip_norm=1.0
    )

    # Predict
    z_test = X_test @ w + b
    y_prob_dp = 1 / (1 + np.exp(-z_test))
    y_pred_dp = (y_prob_dp > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred_dp)
    auc = roc_auc_score(y_test, y_prob_dp)

    results.append({
        'epsilon': eps,
        'accuracy': acc,
        'auc': auc
    })

    privacy_level = "Strong" if eps <= 1 else "Moderate" if eps <= 8 else "Weak"

    print(f"DP Model (ε={eps}, δ={delta}):")
    print(f"  Accuracy: {acc:.1%}")
    print(f"  AUC: {auc:.3f}")
    print(f"  Privacy: {privacy_level}")
    print()

print("✓ All DP models trained\n")

Training DP models...



DP Model (ε=10.0, δ=1e-05):
  Accuracy: 72.6%
  AUC: 0.612
  Privacy: Weak



DP Model (ε=3.0, δ=1e-05):
  Accuracy: 70.8%
  AUC: 0.665
  Privacy: Moderate



DP Model (ε=1.0, δ=1e-05):
  Accuracy: 58.6%
  AUC: 0.465
  Privacy: Strong



DP Model (ε=0.5, δ=1e-05):
  Accuracy: 80.4%
  AUC: 0.534
  Privacy: Strong

✓ All DP models trained



### Compare: Privacy vs Accuracy

In [9]:
# Create comparison dataframe
df_results = pd.DataFrame(results)

# Add baseline
baseline_row = pd.DataFrame([{
    'epsilon': np.inf,
    'accuracy': accuracy_score(y_test, y_pred_baseline),
    'auc': roc_auc_score(y_test, y_prob_baseline)
}])

df_results = pd.concat([baseline_row, df_results], ignore_index=True)

print("COMPREHENSIVE COMPARISON:\n")
print(df_results.to_string(index=False))
print()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy
ax1 = axes[0]
x_labels = ['No DP\n(ε=∞)'] + [f'ε={e:g}' for e in df_results['epsilon'].iloc[1:]]
colors = ['red', 'orange', 'gold', 'green', 'darkgreen'][:len(df_results)]

bars = ax1.bar(range(len(df_results)), df_results['accuracy'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xticks(range(len(df_results)))
ax1.set_xticklabels(x_labels)
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Privacy vs Accuracy Trade-off', fontsize=14, fontweight='bold')
ax1.set_ylim(0.0, 1.0)
majority_rate = max(y_test.mean(), 1 - y_test.mean())
ax1.axhline(majority_rate, color='gray', linestyle='--', label=f'Always predict majority ({majority_rate:.1%})')
ax1.legend(loc='lower left')
ax1.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, df_results['accuracy']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', fontsize=11, fontweight='bold')

# Plot 2: AUC
ax2 = axes[1]
bars = ax2.bar(range(len(df_results)), df_results['auc'], color=colors, edgecolor='black', linewidth=1.5)
ax2.set_xticks(range(len(df_results)))
ax2.set_xticklabels(x_labels)
ax2.set_ylabel('AUC', fontsize=12, fontweight='bold')
ax2.set_title('Privacy vs AUC Trade-off', fontsize=14, fontweight='bold')
ax2.set_ylim(0.0, 1.0)
ax2.axhline(0.5, color='gray', linestyle='--', label='Chance (AUC=0.5)')
ax2.legend(loc='lower left')
ax2.grid(True, alpha=0.3, axis='y')

for bar, auc in zip(bars, df_results['auc']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{auc:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('dp_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Insights computed from the table above (not assumed)
base = df_results.iloc[0]
eps1 = df_results[df_results['epsilon'] == 1.0].iloc[0]
print("\n💡 KEY INSIGHTS (this run):")
print(f"\n1. Majority-class baseline: always predicting 'no diabetes' scores {majority_rate:.1%} accuracy.")
print(f"   The non-private model's {base['accuracy']:.1%} accuracy is {'no better than' if base['accuracy'] <= majority_rate else 'only slightly above' if base['accuracy'] - majority_rate < 0.03 else 'above'} that,")
print(f"   so AUC ({base['auc']:.3f}) is the more meaningful yardstick here.")
print(f"2. ε=1: accuracy {base['accuracy']:.1%} → {eps1['accuracy']:.1%} ({(base['accuracy'] - eps1['accuracy'])*100:.1f} points lost), "
      f"AUC {base['auc']:.3f} → {eps1['auc']:.3f} ({eps1['auc'] - base['auc']:+.3f})")
for _, r in df_results.iloc[1:].iterrows():
    verdict = "no better than chance" if r['auc'] <= 0.55 else "some ranking ability left"
    print(f"   ε={r['epsilon']:g}: AUC {r['auc']:.3f} ({verdict}), accuracy {r['accuracy']:.1%}")
print("3. Accuracy at a fixed 0.5 cut-off mostly reflects where the noisy intercept landed;")
print("   with one noisy run per ε, differences of a few AUC points can be run-to-run noise.")
dp_aucs = df_results.iloc[1:]['auc'].values  # ordered ε = 10, 3, 1, 0.5
if np.any(np.diff(dp_aucs) > 0):
    print("   (Here AUC does not fall monotonically as ε shrinks — a sign of that noise.)")
print("   Exercise: repeat with 5 seeds and plot mean ± SD.")
n_steps = 30 * int(np.ceil(len(X_train) / 64))
print(f"4. Caveat: 'ε' here is the per-step noise setting; {n_steps:,} noisy steps were taken and NOT composed,")
print("   so the trained model has no accounted ε guarantee — Part 2b below does it properly with Opacus.\n")

COMPREHENSIVE COMPARISON:

 epsilon  accuracy      auc
     inf     0.831 0.675059
    10.0     0.726 0.612423
     3.0     0.708 0.664935
     1.0     0.586 0.465248
     0.5     0.804 0.533868




💡 KEY INSIGHTS (this run):

1. Majority-class baseline: always predicting 'no diabetes' scores 83.2% accuracy.
   The non-private model's 83.1% accuracy is no better than that,
   so AUC (0.675) is the more meaningful yardstick here.
2. ε=1: accuracy 83.1% → 58.6% (24.5 points lost), AUC 0.675 → 0.465 (-0.210)
   ε=10: AUC 0.612 (some ranking ability left), accuracy 72.6%
   ε=3: AUC 0.665 (some ranking ability left), accuracy 70.8%
   ε=1: AUC 0.465 (no better than chance), accuracy 58.6%
   ε=0.5: AUC 0.534 (no better than chance), accuracy 80.4%
3. Accuracy at a fixed 0.5 cut-off mostly reflects where the noisy intercept landed;
   with one noisy run per ε, differences of a few AUC points can be run-to-run noise.
   (Here AUC does not fall monotonically as ε shrinks — a sign of that noise.)
   Exercise: repeat with 5 seeds and plot mean ± SD.
4. Caveat: 'ε' here is the per-step noise setting; 1,890 noisy steps were taken and NOT composed,
   so the trained model has no accounted ε 

### Part 2b (optional): Real DP-SGD with Opacus and an RDP accountant

The hand-rolled version above never composed its privacy loss. [Opacus](https://opacus.ai) does the full recipe: per-example clipping, Gaussian noise calibrated to the clipped sum, Poisson-sampled batches, and an **RDP accountant** that composes every step. `PrivacyEngine.make_private_with_epsilon` searches for the noise multiplier that lands exactly on a target (ε, δ) after the chosen number of epochs.

**What the guarantee covers.** The accountant's (ε, δ) applies to the trained model parameters with respect to the **training patients**, under add/remove-one-patient adjacency. It covers the full training pipeline only because preprocessing uses fixed public constants (cell above) and the hyperparameters were not tuned on these data; the test set is never touched in training.

We train the same logistic model (a single `nn.Linear` layer) on the same synthetic diabetes data for ε ∈ {0.5, 1, 3, 10}, δ = 1e-5.

> ℹ️ Needs `torch` and `opacus` (Colab: `pip install opacus`). They are not available in the browser (JupyterLite) version; there this section is skipped and the hand-rolled demo above is the fallback.

In [10]:
try:
    import torch
    from torch import nn
    from torch.utils.data import TensorDataset, DataLoader
    from opacus import PrivacyEngine
    HAS_OPACUS = True
except ImportError:
    HAS_OPACUS = False
    print("ℹ️ This section needs PyTorch + Opacus (Colab or local Jupyter: pip install opacus); it is skipped in the browser version.")

opacus_results = []
if HAS_OPACUS:
    torch.manual_seed(42)
    delta_op = 1e-5           # δ ≪ 1/n
    epochs_op = 10
    Xt = torch.tensor(X_train, dtype=torch.float32)
    yt = torch.tensor(y_train, dtype=torch.float32)
    Xte = torch.tensor(X_test, dtype=torch.float32)

    for eps in [0.5, 1.0, 3.0, 10.0]:
        model = nn.Linear(X_train.shape[1], 1)          # logistic regression
        optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
        loader = DataLoader(TensorDataset(Xt, yt), batch_size=256, shuffle=True)
        engine = PrivacyEngine(accountant='rdp')
        model, optimizer, loader = engine.make_private_with_epsilon(
            module=model, optimizer=optimizer, data_loader=loader,
            target_epsilon=eps, target_delta=delta_op,
            epochs=epochs_op, max_grad_norm=1.0)
        loss_fn = nn.BCEWithLogitsLoss()
        for _ in range(epochs_op):
            for xb, yb in loader:
                if len(xb) == 0:        # Poisson sampling can yield an empty batch
                    continue
                optimizer.zero_grad()
                loss_fn(model(xb).squeeze(-1), yb).backward()
                optimizer.step()
        with torch.no_grad():
            p = torch.sigmoid(model(Xte)).squeeze(-1).numpy()
        spent = engine.get_epsilon(delta_op)
        opacus_results.append({'target ε': eps, 'noise σ': optimizer.noise_multiplier,
                               'accounted ε': spent, 'AUC': roc_auc_score(y_test, p)})

    df_op = pd.DataFrame(opacus_results)
    base_auc = roc_auc_score(y_test, y_prob_baseline)
    print(f"OPACUS DP-SGD (RDP accountant, δ={delta_op}, {epochs_op} epochs):\n")
    print(df_op.round(3).to_string(index=False))
    print(f"\nNon-private baseline AUC: {base_auc:.3f}")
    ok = (df_op['accounted ε'] <= df_op['target ε'] + 1e-6).all()
    print(f"\n💡 The accountant {'confirms' if ok else 'does NOT confirm'} that every model stayed within its target ε —")
    print("   this time ε is an accounted guarantee for the trained weights w.r.t. the training patients")
    print("   (given public preprocessing constants and untuned hyperparameters), not a per-step label.")
    worst = df_op.loc[df_op['AUC'].idxmin()]
    drop = base_auc - worst['AUC']
    if drop < 0.02:
        print(f"   Even at ε={worst['target ε']:g} the AUC loss is only {drop:.3f}: a 3-feature linear model on 4,000")
        print("   patients is easy to privatise. Bigger models / smaller datasets typically pay more.")
    else:
        print(f"   The largest AUC loss is {drop:.3f} at ε={worst['target ε']:g}: stronger privacy costs utility.")
    hand = df_results[df_results['epsilon'] == 1.0]['auc'].iloc[0]
    op1 = df_op[df_op['target ε'] == 1.0]['AUC'].iloc[0]
    print(f"   At ε=1, Opacus AUC {op1:.3f} vs hand-rolled {hand:.3f}: "
          f"{'Opacus is better here' if op1 > hand else 'hand-rolled is not worse here'} — and only Opacus's ε is real.")


W1124 20:42:17.789000 58496 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


OPACUS DP-SGD (RDP accountant, δ=1e-05, 10 epochs):

 target ε  noise σ  accounted ε   AUC
      0.5    6.250        0.500 0.673
      1.0    3.438        0.993 0.675
      3.0    1.494        2.990 0.676
     10.0    0.793        9.995 0.675

Non-private baseline AUC: 0.675

💡 The accountant confirms that every model stayed within its target ε —
   this time ε is an accounted guarantee for the trained weights w.r.t. the training patients
   (given public preprocessing constants and untuned hyperparameters), not a per-step label.
   Even at ε=0.5 the AUC loss is only 0.002: a 3-feature linear model on 4,000
   patients is easy to privatise. Bigger models / smaller datasets typically pay more.
   At ε=1, Opacus AUC 0.675 vs hand-rolled 0.465: Opacus is better here — and only Opacus's ε is real.


## Part 3: Privacy Composition

**Problem**: Privacy degrades with multiple queries/training runs.

In [11]:
print("PRIVACY COMPOSITION EXAMPLE:\n")
print("Scenario: Hospital wants to release 10 statistical queries\n")

# Single query
epsilon_per_query = 0.1
n_queries = 10

print(f"Privacy per query: ε = {epsilon_per_query}")
print(f"Number of queries: {n_queries}\n")

level = lambda e: "Strong (ε ≤ 1)" if e <= 1 else "Moderate (1 < ε ≤ 8)" if e <= 8 else "Weak (ε > 8)"

# Sequential composition (basic)
total_epsilon_basic = epsilon_per_query * n_queries

print(f"Basic Composition:")
print(f"  Total privacy loss: ε = {total_epsilon_basic}")
print(f"  Privacy level: {level(total_epsilon_basic)}\n")

# Advanced composition (Dwork-Rothblum-Vadhan): (ε', kδ + δ')-DP with
# ε' = sqrt(2k ln(1/δ')) ε + k ε (e^ε − 1)
delta = 1e-5
def advanced_composition(eps, k, delta_prime):
    return np.sqrt(2 * k * np.log(1 / delta_prime)) * eps + k * eps * (np.exp(eps) - 1)
total_epsilon_advanced = advanced_composition(epsilon_per_query, n_queries, delta)

print(f"Advanced Composition (δ' = {delta}):")
print(f"  Total privacy loss: ε ≈ {total_epsilon_advanced:.2f}")
print(f"  Privacy level: {level(total_epsilon_advanced)}")
if total_epsilon_advanced >= total_epsilon_basic:
    print(f"  → For only {n_queries} queries the 'advanced' bound is LOOSER than basic composition;")
    print("    always report the smaller of the two.")
k_big = 1000
print(f"  For k={k_big} queries: basic ε = {epsilon_per_query * k_big:.0f}, "
      f"advanced ε ≈ {advanced_composition(epsilon_per_query, k_big, delta):.1f} "
      "(advanced composition pays off when k is large)\n")

print("💡 Implication: Privacy budget must be carefully managed across:")
print("   • Data exploration")
print("   • Model training")
print("   • Hyperparameter tuning")
print("   • Model evaluation")
print("   • Publication of results")
print("\n   Once budget is exhausted, no more queries allowed!\n")

PRIVACY COMPOSITION EXAMPLE:

Scenario: Hospital wants to release 10 statistical queries

Privacy per query: ε = 0.1
Number of queries: 10

Basic Composition:
  Total privacy loss: ε = 1.0
  Privacy level: Strong (ε ≤ 1)

Advanced Composition (δ' = 1e-05):
  Total privacy loss: ε ≈ 1.62
  Privacy level: Moderate (1 < ε ≤ 8)
  → For only 10 queries the 'advanced' bound is LOOSER than basic composition;
    always report the smaller of the two.
  For k=1000 queries: basic ε = 100, advanced ε ≈ 25.7 (advanced composition pays off when k is large)

💡 Implication: Privacy budget must be carefully managed across:
   • Data exploration
   • Model training
   • Hyperparameter tuning
   • Model evaluation
   • Publication of results

   Once budget is exhausted, no more queries allowed!



## Summary and Key Takeaways

### What We Learned

1. **Differential Privacy Basics**:
   - Mathematical guarantee: individual's data doesn't significantly affect output
   - ε (epsilon) = privacy budget: smaller = stronger privacy
   - Achieved by adding calibrated noise (Laplace for counting, Gaussian for ML)

2. **DP for Machine Learning**:
   - **DP-SGD**: Clip gradients + add noise during training
   - Privacy-utility trade-off: measure it on your own data (in Part 2 it was large; judge by AUC, not accuracy, when classes are imbalanced)
   - Our simplified DP-SGD did no privacy accounting; Part 2b used Opacus's RDP accountant to state a true end-to-end ε

3. **Privacy Composition**:
   - Multiple queries consume privacy budget additively
   - Must allocate budget across entire ML pipeline
   - Advanced composition gives tighter bounds only when the number of queries is large

4. **Practical Guidance**:
   ```
   Recommended ε for healthcare:
   - Research: ε = 1-8 (balances privacy + utility)
   - Clinical deployment: ε = 0.5-1 (stronger privacy)
   ```

### Best Practices

✅ **Use DP for sensitive datasets**: Medical records, genomics, mental health  
✅ **Prefer small ε (≤1 is commonly called strong)**: patient trust  
✅ **Clip gradients carefully**: Too large → privacy leak; too small → slow convergence  
✅ **Track privacy budget**: Don't exceed allocated ε across all experiments  
✅ **Communicate to patients**: "Your data contributes to research with mathematical privacy guarantee"  

### Limitations

⚠️ **Accuracy degradation**: Strong privacy (ε<1) can cause 10-20% accuracy loss  
⚠️ **Hyperparameter sensitivity**: DP-SGD requires careful tuning (clip norm, noise scale)  
⚠️ **Computational cost**: 2-5× slower training due to per-example gradient clipping  
⚠️ **Limited to training data**: DP doesn't protect against model inversion at inference  

### Real-World Impact

**Success Stories**:
- **Apple**: DP for keyboard suggestions, emoji usage (ε=4-8)
- **Google**: DP for Chrome telemetry, federated learning (ε=1-10)
- **US Census 2020**: DP for demographic statistics (ε=19.6 total)

**Healthcare Applications**:
- Multi-hospital collaborations with DP guarantees
- Public release of synthetic medical datasets
- Privacy-preserving AI research

---

**Next**: Notebook 11.2 covers Federated Learning—collaborative training without data sharing.

*"Privacy is not about hiding bad behavior. It's about protecting individual dignity."*